# Pac-Man con Redes Neuronales + AlphaBeta

## Autores: Ismael Escribano Orts y Marcos Hernández Juárez

### 0. Estructura de la entrega
```cpp
entrega.zip
├── pacman_model.pth    // Modelo de la Red Neuronal de Pac-Man
├── multiAgents.py      // Clases NeuralAgent, AlphaBetaAgent y AlphaBetaNeuralAgent
├── pacman.py           // Modificación para indicar la semilla
├── runner.py           // Función auxiliar para ejecutar partidas
├── memoria.ipynb       // Memoria de la práctica
├── videos/             // Directorio con los videos de las mejores partidas
```

### 1. *Introducción*

El objetivo de esta práctica se divide en dos partes, aunque ámbas relacionadas con poder jugar a Pac-Man de manera autónoma y eficiente:

1. Entrenamiento de la Red Neuronal (NN): Se ha necesitado jugar manualmente partidas para posteriormente entrenar una red neuronal que nos permita jugar de manera automática, aunque sin seguir ninguna estrategia, simplemente aprendiendo de nuestros moviemientos para después replicar nuestro comportamiento en el juego.

2. Implementación de estrategia: Adicionamente, es necesario dotar al agente de estrategia, para ello se usaría el algoritmo Minimax, pero debido a su coste computacional, utilizaremos la versión de AlphaBeta. Esto nos va a permitir añadirle un mejor criterio a la red y conseguir mejores resultados globales.

### 2. *Nuevas heurísticas*

#### 2.1. *Heurísticas iniciales*

Tanto la NN como el algoritmo AlphaBeta van a trabajar según unas heurísticas dadas, que será nuestra estrategia, en el código dado tenemos las siguientes heurísticas iniciales:
```python
# Factor 1: Distancia a la comida más cercana
if food:
    min_food_distance = min(manhattanDistance(pacman_pos, food_pos) for food_pos in food)
    score += 1.0 / (min_food_distance + 1)

# Factor 2: Proximidad a fantasmas
for ghost_state in ghost_states:
    ghost_pos = ghost_state.getPosition()
    ghost_distance = manhattanDistance(pacman_pos, ghost_pos)
    
    if ghost_state.scaredTimer > 0:
        # Si el fantasma está asustado, acercarse a él
        score += 50 / (ghost_distance + 1)
    else:
        # Si no está asustado, evitarlo
        if ghost_distance <= 2:
            score -= 200
```

#### 2.2 *Nuestras heurísticas*
Estas heurísticas son muy sencillas y simplemente tienen en cuenta casos simples, en primer lugar, si nos acercamos a la comida, ganamos más puntos, y en segundo lugar, evitaremos acercarnos a los fantasmas siempre y cuando no esten asustados, en ese caso, el objetivo será acercarse.

##### 2.2.1 **Distancia a la comida más cercana**

En nuestro caso, hemos querido modificar el comportamiento inical de la red y añadir nuevos factores, en primer lugar, tenemos el primer factor ligeramente modificado:
```py
if food:
    min_food_distance = min(manhattanDistance(pacman_pos, food_pos) for food_pos in food)
    score += 10 / (min_food_distance + 1)
    score -= 50 * len(food)
```
En esta primera heurística hemos hecho dos modificaciones, en primer lugar le hemos dado más valor a la comida, debido a los otros factores, tenemos que compensar la comida para evitar que el agente se quede bloqueado. Por otro lado, penalizamos en gran medida la comida que quede, haciendo que Pac-Man prefiera siempre comer.

##### 2.2.2. **Comportamiento contra fanstasmas**

Para el segundo factor, hemos decidido modificar el comportamiento original:
```py
for ghost_state in ghost_states:
    ghost_pos = ghost_state.getPosition()
    ghost_distance = manhattanDistance(pacman_pos, ghost_pos)
    
    if ghost_state.scaredTimer > 0:
        if ghost_distance <= 2:
            score -= 200 
        elif ghost_distance > ghost_state.scaredTimer:
            score -= 50 / (ghost_distance + 1)
    else:
        score -= 100 / (ghost_distance + 1)
```

Aunque el objetivo del agente es obtener la mayor puntuación posible, hemos decidido que el agente tome decisiones más seguras, que aunque lleven a puntuaciones más bajas, llevan a un ratio de victorias mayor, y personalmente, consideramos que ganar con un puntaje medio es mejor que perder con uno alto (Además, ganar da +500 puntos y perder -500). Para conseguir esto, hemos realizado la siguiente lógica: Si el fantasma está asustado, no queremos que comerlo sea una prioridad, por tanto vamos a perjudicar mucho si Pac-Man está cerca de él, además, queremos que mantenga una distancia segura, es decir, si el fantasma tiene el estado asustado 5 turnos más, Pac-Man debe mantenerse aproximadamente a esa misma distancia. Por otro lado si el fantasma no está asustado, seguimos evitandolo, pero a diferencia del factor original, hemos usado un gradiente, para penalizar más o menos según la distancia a la que se encuentra.

En definitiva, con esta heurística buscamos que el tiempo asustado sea un "*periodo de paz*" que nos permita acercanos a la victoria, aunque no queremos que Pac-Man sea totalmente "*pacifista*" ya que para ello tendríamos que hacer medidas más extremas que creemos que no son eficientes, ya que esto podría hacer que el agente evite acercarse a los fantasmas asustados aunque haya comida cerca y prefiera quedarse en una zona vacía perdiendo tiempo (y por tanto puntos), haciendo que cuando los fantasmas sean peligrosos de nuevo tenga que comer más.

##### 2.2.3 **Comer durante el "*periodo de paz*"**

A partir de ahora, continuamos con los factores añadidos por nosotros, primero, el tercer factor, priorizar comida cuando los fantasmas están asustados:
```py
times = []
for ghost_state in ghost_states:
    times.append(ghost_state.scaredTimer)

if all(times) and food:
    min_food_distance = min(manhattanDistance(pacman_pos, food_pos) for food_pos in food)
    score += 20 / (min_food_distance + 1)
```

En primer lugar, comprobamos que todos los fantasmas estén asustados, ya que si alguno no lo está, está heurística daría problemas, ya que compensaría parte de la penalización por cercanía a los fantasmas, haciendo que realice moviemientos arriesgados, lo cual es justamente lo contrario a nuestro objetivo. Entonces, si todos los fantasmas están asustados, vamos a beneficiar en gran medida el acercarnos a la comida, ya que este tiempo es el "*periodo de paz*" del que hablamos en la anterior heurística.

##### 2.2.4. **Comportamiento defensivo con las capsulas**

A continuación, el cuarto factor, la estrategía defensiva de comer capsulas si percibimos peligro:
```py
if capsules:
    min_capsule_distance = min(manhattanDistance(pacman_pos, capsule_pos) for capsule_pos in capsules)
    fantasmas_cerca = False
    for ghost_state in ghost_states:
        ghost_pos = ghost_state.getPosition()
        ghost_distance = manhattanDistance(pacman_pos, ghost_pos)

        if ghost_distance <= 4 and ghost_state.scaredTimer == 0:
            fantasmas_cerca = True
            break
        
    if fantasmas_cerca:
        if min_capsule_distance <= 3:
            score += 50 / (min_capsule_distance + 1)
    else:
        if min_capsule_distance <= 3:
            score += 5 / (min_capsule_distance + 1)
    score -= 20 * len(capsules)
```

En esta heurística, buscamos activar el "**modo defensivo**" del agente, en primer lugar comprobamos si hay fantasmas cerca, si los hay y estamos cerca de una capsula, vamos a hacer que el agente vaya a por ella con una puntuación elevada, si no, le damos uan bonificación mucho menor, ya que, como en la primera heurística, acercarse a la comida es bueno, pero en el caso de las capsulas no es tan útil. Además esta heurística solo se activa si se encuentra cerca de una capsula, si esta siendo perseguido por un fantasma no queremos obligar al agente a ir hacia la capsula ya que podría considerarse un camino peligroso.
Por otro lado, al igual que en la primera heurística, penalizamos la cantidad de capsulas que quedan por comer.

##### 2.2.5. **Terminar la partida cuando queda poca comida**

El siguente es el quinto factor, priorizar terminar la partida:
```py
if food and len(food) <= 15:
    min_food_distance = min(manhattanDistance(pacman_pos, food_pos) for food_pos in food)
    score += 20 / (min_food_distance + 1)
```

Cuando queda poca comida (para este caso, hemos decidido 15 de comida), queremos evitar el Pac-Man se quede atascado esquivando fantasmas o recorriendo el mapa sin objetivos, por tanto, buscamos darle más valor a la comida y por tanto, priorizar acabar la partida lo antes posible.

#### 2.2.6 **Penalizar comida lejana**

La última heurística, el sexto factor, penalizar la comida lejana:
```py
if food:
    min_food_distance = min(manhattanDistance(pacman_pos, food_pos) for food_pos in food)
    if min_food_distance >= 8:
        score -= 1.5 * min_food_distance
```

Esta última heurística se ha añadido para evitar que el agente se mantenga en situaciones suboptimas, es decir, lejos de la comida, aceptamos que el agente se aleje temporalmente de la comida si decide que es la mejor opción, pero en algunas pruebas realizadas sin esta heurística, este problema no era temporal o tenía una duración muy elevada, por tanto, si Pac-Man se encuentra lejos de la comida, será penalizado ligeramente, haciendo que se deba acercar.

**En resumen**, nos hemos centrado en un comportamiento neutral, no buscamos que Pac-Man sea muy agresivo, pero tampoco creemos que jugar defensivo sea la mejor opción, por tanto con estas heurísticas hemos buscado un estilo de juego intermedio, donde evite realizar jugadas arriesgadas, pero no dude en hacerlas si es necesario, y, en general, buscar preferiblemente la victoria en vez de tener romper el saco de la *avaricia* buscando más puntos.

### 3. *Entrenamiento de la red*

### 4. *NN + AlphaBeta*

En esta parte, vamos a juntar la NN previamente entrenada y añadirle el algoritmo AlphaBeta, pero antes de empezar, ¿Qué es AlphaBeta?

#### 4.1. *Algoritmo MiniMax / AlphaBeta*

El algoritmo *Minimax* es un algoritmo de toma de decisiones utilizado principalmente en juegos. La clave de este algoritmo son los dos jugadores: el jugador (Pac-Man/MAX) y su oponente (MIN/Fantasma), **MAX** busca obtener la mejor puntuación posible y **MIN** busca que **MAX** tenga la peor puntuación posible, para ello se codifica los movimientos del juego en un arbol, el cual normalmente se explorará en profundidad, se obtendrá un valor al llegar a la profundidad limite o a un nodo hoja, en ese caso se evalua la puntuación de **MAX**, si en ese momento el turno es de **MAX**, se obtiene la puntuación del hijo mayor, si el turno es de **MIN**, el contrario, se obtiene la puntuación del hijo menor, estas puntuaciones se propagan hacia arriba hasta llegar a la raíz, que siempre será **MAX** y en ese momento podrá decidir cual es el mejor movimiento en ese punto.

Aunque *Minimax* es un algoritmo funcional en la teoría, en la práctica es totalmente ineficiente, para Pac-Man, cada jugador tiene 4 movimientos (Norte, Este, Oeste, Sur), y de cada movimiento aparecen otros 4 del siguiente jugador, además, la profundidad del algoritmo solo aumenta cuando han jugado 1 turno todos los jugadores, es decir, en un mapa con Pac-Man y dos fantasmas, por tanto, en un nivel de profundidad tendríamos un factor de ramificación (b) de 4 (4 movimientos) y una profundidad del arbol (d) de 3. Podemos obtener la cantidad de nodos de un nivel con $b^d$ y el número total de nodos con $\frac{b^{d+1} - 1}{b - 1}$, por tanto un nivel de profundidad tiene $\frac{4^3 - 1}{3 - 1} = 85$ nodos.

Hemos implementado la clase `MiniMaxAgent` que nos permite jugar a pacman con el algoritmo *Minimax* por defecto con una profundidad máxima de 2 para evitar tiempos de ejecución elevados, hemos recibido el código ya hecho.

Para solucionar la ineficiencia del algoritmo *Minimax* existe la poda *AlphaBeta* la cual os permite reducir significativamente la cantidad de nodos explorados sin afectar al resultado final, esto se consigue eliminando ramas que no van a influenciar en la decisión final, para ello necesitamos dos nuevos valores:

- Alpha ($\alpha$) -> Valor máximo encontrado por el jugador MAX
- Beta ($\beta$) -> Valor minimo encontrado por el jugador MIN

Recordamos que, el algoritmo funciona desde el punto de vista de MAX, por tanto, $\beta$ es la puntuación minima de MAX que ha encontrado MIN. Además, consideramos que MIN siempre juega la mejor jugada posible, aunque luego realmente no lo haga.

La poda es muy simple, cortamos una rama si $\alpha \ge \beta$, ya que, si juega MAX y la condición se cumple, MIN nunca va a permitir que vayas por esa rama (porque elegirá una peor). Por otro lado, si juega MIN y la condición se cumple es al contrario, MAX nunca va a permitir ir por esa rama (porque elegirá una mejor).

Finalmente, hemos implementado la clase `AlphaBetaAgent`, con la misma estructura que `MiniMaxAgent`, el algoritmo es muy similar con ligeros cambios ya que hay que añadir la poda:
```py
def alphabeta(gameState, depth, alpha, beta, agentIndex):
    if depth == self.depth or gameState.isWin() or gameState.isLose():
        return self.evaluationFunction(gameState)
    
    if agentIndex == 0: # Turno de Pacman (MAX)
        max_eval = float('-inf')
        actions = gameState.getLegalActions(agentIndex)
        if not actions:
            return self.evaluationFunction(gameState)
        for action in actions:
            successor = gameState.generateSuccessor(agentIndex, action)
            eval_score = alphabeta(successor, depth, alpha, beta, agentIndex+1)
            max_eval = max(max_eval, eval_score)
            alpha = max(alpha, eval_score)
            if beta <= alpha:
                break
        return max_eval
    
    else: # Turno de los fantasmas (MIN)
        min_eval = float('inf')
        actions = gameState.getLegalActions(agentIndex)
        if not actions:
            return self.evaluationFunction(gameState)
        
        nextAgent = agentIndex + 1
        nextDepth = depth
        if nextAgent == gameState.getNumAgents():
            nextAgent = 0
            nextDepth = depth + 1
        
        for action in actions:
            successor = gameState.generateSuccessor(agentIndex, action)
            eval_score = alphabeta(successor, nextDepth, alpha, beta, nextAgent)
            min_eval = min(min_eval, eval_score)
            beta = min(beta, eval_score)
            if beta <= alpha:
                break
        return min_eval
```

#### 4.2. *Integración NN +  AlphaBeta*

Ya tenemos la Red Neuronal y el algoritmo *AlphaBeta* preparados, ahora podemos comenzar a integrar ámbos a la vez para tener un agente neuronal con estrategia implementada. Para ello hemos implementado la clase `AlphaBetaNeuralAgent`, para aprovechar la reutilización de código, hemos hecho que herede de `NeuralAgent` y así solo tenemos que implementar de nuevo el algoritmo *AlphaBeta* (lo cual está bien porque debemos realizar una pequeña modificación). En primer lugar, la construcción de la instancia del agente es la siguiente:
```py
class AlphaBetaNeuralAgent(NeuralAgent):
    def __init__(self, model_path="models/pacman_model.pth", depth = '3', w_trad = 0.25, w_neural = 0.75):
        super().__init__(model_path)

        self.depth = int(depth)
        
        self.w_trad = w_trad
        self.start_trad = w_trad
        self.end_trad = 1 - w_trad

        self.w_neural = w_neural
        self.start_neural = w_neural
        self.end_neural = 1 - w_neural

        self.total_food = None
```

Parámetros:
- depth: Profundidad máxima del algoritmo *AlphaBeta*, a mayor profundidad, los resultados deberían ser mejores, pero el tiempo de ejecución aumenta exponencialmente.
- w_trad: Peso inicial para las heurísticas tradicionales. (Punto 2.)
- w_neural: Peso inicial para la red neuronal.

Adicionalmente se crean los siguientes atributos de instancia:
- start_trad: Valor inicial del peso de las heurísticas tradicionales.
- end_trad: Valor final del peso de las heurísticas tradicionales.
- start_neural y end_neural: Igual que las anteriores, pero para la red neuronal.
- total_food: Comida total del mapa, será explicado más adelante.

Estos atributos extra son necesarios ya que hemos implementado pesos dinamicos según el progreso de la partida, nuestro pensamiento es que las partidas de entrenamiento de la red neuronal inician de una manera similar, por tanto, su inicio es muy confiable, a medida que la partida avanza, sus movimientos son más erráticos e improvisados, es decir, no podemos confiar tanto en ella. Por otro lado, creemos que las heurísticas tienen un buen desempeño en todo momento, asique a medida que la red se hace menos confiable, le damos esa confianza a las heurísticas.

Como hemos dicho anteriormente, el algoritmo *AlphaBeta* es el mismo pero con una pequeña modificación, originalmente se utiliza `evaluationFunction()` para evaluar los estados, ahora usará `combined_evaluation()`, que es la combinación entre las heurísticas y la red neuronal:
```py
def combined_evaluation(self, state):
    trad_score = self.evaluationFunction(state, evaluation=Evaluation.heuristics)

    neural_score = self.evaluationFunction(state, evaluation=Evaluation.neural)
    
    return self.w_trad * trad_score + self.w_neural * neural_score
```

En esta función, `evaluationFunction` es la función de evaluación de `NeuralAgent`, que contiene la lógica para obtener la puntuación del estado según la red neuronal y según las heurísticas introducidas, el parametro "evaluation" viene de `class Evaluation(Enum)` que nos permite ejecutar únicamente las heurísticas, únicamente la puntuación neural o ambas, esta última usada unicamente para el `NeuralAgent` puro (funcionamiento original).

Las acciones se obtienen de la siguiente manera:
```py
def getAction(self, state: GameState):
    best_score = float('-inf')
    best_action = None
    for action in state.getLegalActions(0):
        succesor = state.generateSuccessor(0, action)
        score = self.alphabeta(succesor, 0, float('-inf'), float('+inf'), 1)
        if action == Directions.STOP:
            score -= 200
        if score > best_score:
            best_score = score
            best_action = action
    # Obtenemos la comida total en el primer turno
    if self.total_food is None:
        self.total_food = state.getNumFood()
    
    # Obtenemos el progreso del juego (% de comida por comer)
    current_food = state.getNumFood()
    game_progress = current_food / self.total_food

    # Modificamos los pesos usando interpolación lineal
    self.w_neural = self.start_neural + (self.end_neural - self.start_neural) * game_progress
    self.w_trad = self.start_trad + (self.end_trad - self.start_trad) * game_progress
    return best_action
```

Ejecutamos alphabeta con los valores iniciales, además, añadimos una penalización si la acción es "STOP", esto se debe a que quedarse quieto es perder el tiempo. En algunos momentos el agente espera a las acciones de los fantasmas para decidir que hacer, realizar "STOP" en esos casos es correcto, pero el problema está en que si no lo penalizamos, lo hará más de lo que realmente debería, por tanto, preferimos que cuando deba esperar tenga que moverse hacia los lados en vez de pararse.

Por otro lado, en el primer turno obtenemos la comida total, para poder obtener el progreso del juego, que para nosotros será el porcentaje de comida por comer, este progreso es necesario para realizar una interpolación lineal de los pesos, haciendo que a medida que avanza el juego, los pesos pasan de sus valores "*start*" a sus valores "*end*". La fórmula general de la interpolación lineal es la siguiente:

$$
start + (end - start) * progress
$$

Creemos que usar este método mátematico para los pesos es una gran idea porque evitamos tener que normalizar pesos (siempre van a estar dentro de los valores que queremos) además de que nos permite tener una progresión de los pesos lineal, ya que realizar acciones como "añadir/reducir un 5% por acción" puede modificar demasiado rápido los pesos aunque el agente realmente no este avanzando en el juego.

### 5. *Resultados*

Para ejecutar las pruebas necesarias, la interfaz de pacman.py nos estaba dando problemas, sobretodo para obtener la semilla de las partidas, por tanto hemos decidido hacer un pequeño script que se encargue de ejecutarlas exactamente como queremos, el archivo es `runner.py`, en la parte de "Configuración general" se puede modificar el comportamiento. Cabe indicar que el script esta diseñado para trabajar con `AlphaBetaNeuralAgent`, si se busca usar otro es necesario modificar alguna parte del código. Además, durante la ejecución de las partidas si se realiza "Control+C" se muestra un resumen de las partidas terminadas, aunque no todas se hayan terminado

Las pruebas se han ejecutado con una profundidad máxima de 5 (Valor elevado, pero los resultados son buenos) las semillas utilizadas son de la 0-9 (10 partidas en total) y los pesos iniciales son: 0.25 a las heurísticas tradicionales y 0.75 a la red neuronal.

A continuación indicamos las puntuaciones obtenidas y su valor medio:

In [3]:
import numpy as np

# cl -> Classical Layout | nl -> New Layout

# NN
neural1_cl = [-374.0, -384.0, -52.0, -352.0, -289.0, 159.0, -365.0, -296.0, -442.0, -426.0]
neural1_nl = [-89.0, -473.0, -366.0, -415.0, -289.0, -152.0, -365.0, -264.0, -449.0, -390.0]

print(f'Mean score for NN in cl: {np.mean(neural1_cl)}')
print(f'Mean score for NN in nl: {np.mean(neural1_nl)}')
print('=' * 60)

# NN + new heuristics
neural2_cl = [-311.0, -374.0, -326.0, -160.0, -136.0, -425.0, -380.0, -360.0, -414.0, -340.0]
neural2_nl = [-349.0, -374.0, -370.0, 188.0, 127.0, -425.0, -370.0, -360.0, -414.0, -395.0]

print(f'Mean score for NN + heuristics in cl: {np.mean(neural2_cl)}')
print(f'Mean score for NN + heuristics in nl: {np.mean(neural2_nl)}')
print('=' * 60)

# AlphaBeta + NN + new heuristics
ab_neural_cl = [1729.0, 1701.0, 1795.0, 2127.0, 1913.0, 467.0, 398.0, 891.0, 947.0, 1909.0]
ab_neural_nl = [1568.0, 2144.0, 1726.0, 1753.0, 508.0, 932.0, 1929.0, 1762.0, 1921.0, 714.0]

print(f'Mean score for AlphaBeta + NN + heuristics in cl: {np.mean(ab_neural_cl)}')
print(f'Mean score for AlphaBeta + NN + heuristics in nl: {np.mean(ab_neural_nl)}')


Mean score for NN in cl: -282.1
Mean score for NN in nl: -325.2
Mean score for NN + heuristics in cl: -322.6
Mean score for NN + heuristics in nl: -274.2
Mean score for AlphaBeta + NN + heuristics in cl: 1387.7
Mean score for AlphaBeta + NN + heuristics in nl: 1495.7


| Configuration                          | mediumClassic               | customMaze                 |
|----------------------------------------|-----------------------------|----------------------------|
| Greedy neural agent (no modifications) | score: -282.1 winrate: 0%   | score: -325.2 winrate: 0%  |
| Greedy neural agent + new heuristics   | score: -322.6 winrate: 0%   | score: -274.2 winrate: 0%  |
| AlphaBeta + NN + new heuristics        | score: 1387.7 winrate: 60%  | score: 1495.7 winrate: 70% |
